<a href="https://colab.research.google.com/github/velosov/2bak/blob/main/QEA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
import requests
import os
from math import sqrt, sin, cos, pi, asin, acos, e
import cmath
import numpy as np
import json
import io
from random import randint, uniform

In [ ]:
res = requests.get('https://drive.google.com/uc?id=1KooJzeuq0yRSdcT1OWwbKGR3VgOsvsDZ')
assinaturas = res.json()
print(assinaturas['blackout'])
#print([[variavel[0] for variavel in ] for acidente in assinaturas])
# with io.open('/content/drive/MyDrive/TCC/assinaturas.json', 'r', encoding='utf-8-sig') as data:
#     dct = json.load(data)
#     print(dct)

[{'vazDoNucleo': '+.105223E+03', 'tempPQuente': '+.324465E+03', 'tempPFria': '+.291456E+03', 'vazNoNucleo': '+.504184E+05', 'GVLarga': '+.500000E+02', 'GVEstreita': '+.625153E+02', 'pressaoGV': '+.689480E+01', 'vazAguaAlimentacao': '+.527767E+03', 'vazVapor': '+.527767E+03', 'vazRuptura': '+.000000E+00', 'vazPrimario': '+.105223E+03', 'tempo': '+.000000E+00', 'pressaoPrimario': '+.158000E+02', 'potTermica': '+.100000E+03', 'potNuclear': '+.100000E+03', 'subresfriamento': '+.218620E+02', 'nivelPressurizador': '+.591491E+02', 'tempPrimario': '+.307959E+03'}, {'vazDoNucleo': '+.105390E+03', 'tempPQuente': '+.324463E+03', 'tempPFria': '+.291457E+03', 'vazNoNucleo': '+.504183E+05', 'GVLarga': '+.500078E+02', 'GVEstreita': '+.625169E+02', 'pressaoGV': '+.689413E+01', 'vazAguaAlimentacao': '+.527890E+03', 'vazVapor': '+.528396E+03', 'vazRuptura': '+.000000E+00', 'vazPrimario': '+.105390E+03', 'tempo': '+.100000E+01', 'pressaoPrimario': '+.157999E+02', 'potTermica': '+.100005E+03', 'potNuclear

In [ ]:
a = np.array([[1,2],[3,4]])
print(type(a))
b = np.array([3,4])
b1 = np.array((3,4))
c = np.array([[0,1],[1,0]])

print(np.dot(a,b))
print(np.dot(a,b1))
print(np.dot(c,a))
print(a)

<class 'numpy.ndarray'>
[11 25]
[11 25]
[[3 4]
 [1 2]]
[[1 2]
 [3 4]]


In [21]:
class Qubit():
    def __init__(self,cromo, id, semente=False):
        self.gene = self.semear(semente)
        self.alelo = self.colapsar()
        self.cromo = cromo
        self.id = id

    def semear(self,semente):
        sementes = {
            'real': [1/sqrt(2),1/sqrt(2)],
            'complexa': [1j/sqrt(2),1j/sqrt(2)]
        }
        if semente in sementes:
            return sementes[semente]
        return np.array(sementes['real'])

    def colapsar(self):
        if uniform(0,1) > (self.gene[0])**2:
            return 1
        return 0

    def operadores(self,portao,teta):
        transformacoes = {
                'PauliX': lambda teta : [[0,1],[1,0]],
                'PauliY': lambda teta : [[0,-1j],[1j,0]],
                'PauliZ': lambda teta : [[1,0],[0,-1]],
                'H' : lambda teta : [[1/sqrt(2),1/sqrt(2)],[1/sqrt(2),-1/sqrt(2)]],     #na Esfera de Bloch, carateriza uma rotação de 90o em torno do eixo y seguida de uma rotação de 180◦ em torno do eixo z .
                'S' : lambda teta : [[1,0],[0,1j]],                #a porta S, na Esfera de Bloch, se comporta como uma rotação de 90o em torno do eixo z
                'T' : lambda teta : [[1,0],[0,e**(pi*4*1j)]],       #a porta T representa uma rotação de 45o em torno do eixo z da esfera
                'Rx': lambda teta : [[cos(teta/2), -1j*sin(teta/2)],[-1j*sin(teta/2), cos(teta/2)]],
                'Ry': lambda teta : [[cos(teta/2), -sin(teta/2)],[sin(teta/2), cos(teta/2)]],
                'Rz': lambda teta : [[e**(-1j*teta/2), 0],[0, e**(-1j*teta/2)]]
                }
        return np.array(transformacoes[portao](teta))

    def atualizar(self, fitness, portao, delta, epsilon, maiorScore, melhor):
        if self.alelo == melhor['fenotipos'][self.id]:
            return
        alfa = self.gene[0]
        beta = self.gene[1]
        #Checa se alfa ou beta são < epsilon, caso onde He intervém
        if (alfa or beta) ** 2 < epsilon:
            self.gene = np.array([sqrt(epsilon), sqrt(1-epsilon)])
            return
        # if beta ** 2 < epsilon:
        #     self.gene = [sqrt(1-epsilon),sqrt(epsilon)]
        #     return
        #Tabela de Aplicação do Q-Gate
        if fitness >= maiorScore:
            if self.alelo == 0:
                direction = 0
            else:
                direction = 1
        else:
            if self.alelo == 0:
                direction = 1
            else:
                direction = 0
        if alfa*beta >0:
            if direction==1:
                s = 1
            else:
                s = -1
        else:
            if direction==1:
                s = -1
            else:
                s = 1
        teta = s*delta
        transformacao = self.operadores('Ry',teta)
        self.gene = np.dot(transformacao,self.gene)
        self.alelo = self.colapsar()
        return

    def props(self):
        resposta = {
            'id': self.id,
            'cromo': self.cromo,
            'gene': [self.gene[0],self.gene[1]],
            'alelo': self.alelo,
        }
        return resposta

In [ ]:
operadores = {
                'PauliX': lambda teta : [[0,1],[1,0]],
                'PauliY': lambda teta : [[0,-1j],[1j,0]],
                'PauliZ': lambda teta : [[1,0],[0,-1]],
                'H' : lambda teta : [[1/sqrt(2),1/sqrt(2)],[1/sqrt(2),-1/sqrt(2)]],     #na Esfera de Bloch, carateriza uma rotação de 90o em torno do eixo y seguida de uma rotação de 180◦ em torno do eixo z .
                'S' : lambda teta : [[1,0],[0,1j]],                #a porta S, na Esfera de Bloch, se comporta como uma rotação de 90o em torno do eixo z
                'T' : lambda teta : [[1,0],[0,e**(pi*4*1j)]],       #a porta T representa uma rotação de 45o em torno do eixo z da esfera
                'Rx': lambda teta : [[cos(teta/2), -1j*sin(teta/2)],[-1j*sin(teta/2), cos(teta/2)]],
                'Ry': lambda teta : [[cos(teta/2), -sin(teta/2)],[sin(teta/2), cos(teta/2)]],
                'Rz': lambda teta : [[e**(-1j*teta/2), 0],[0, e**(-1j*teta/2)]]
                }

print(operadores['H'](pi))

[[0.7071067811865475, 0.7071067811865475], [0.7071067811865475, -0.7071067811865475]]


In [ ]:
#print(Qubit.semear('complexa'))
print(uniform(0,1))

epsilon = 0.1
alfa = 0.09
beta = 1-alfa
ab={'alelo': 1}
bc={'alelo': 0}
if (alfa or beta) ** 2 < epsilon:
            print('y')
print([i['alelo'] for i in [ab,bc] ])


0.9774512067215154
y
[1, 0]


In [23]:
class Cromossomo():
    def __init__(self, id, geracao=0, numAcidentes=3, numVariaveis=18, bitsPorVar=12):
        self.id = id
        self.numVariaveis = numVariaveis
        self.bitsPorVar = bitsPorVar
        self.dna = [Qubit(id,alelo) for alelo in range(numAcidentes*numVariaveis*bitsPorVar)]
        self.fenotipo = self.decodificar()
        self.prototipos = self.transcrever()
        self.fitness = 0

    def decodificar(self):
        return [qubit.alelo for qubit in self.dna]

    def decimalize(self, binList):
        #binário de Gray é convertido para decimal com bit representando coeficiente de serie de potência base 1/2
        coeffs = []
        for i in range(len(binList)):
            coeffs.append(binList[i] * (2**(-(i+1))))
        return sum(coeffs)

    def gray(self,binsList):
        #Gray Code: aplica XOR ao bit com adjacente à esquerda até msb
        grays = []
        for binList in binsList:
            xorOutput = []
            mostSigBit = binList.pop(0)
            outrosBits = binList[::-1]
            for i in range(len(outrosBits)):
                if i==(len(outrosBits)-1):
                    comp = mostSigBit
                else:
                    comp = outrosBits[i+1]

                if outrosBits[i] == comp:
                    xorOutput.append(0)
                else:
                    xorOutput.append(1)
            xorOutput.append(int(mostSigBit))
            grays.append(xorOutput[::-1])
        return [self.decimalize(bin) for bin in grays]

    def transcrever(self):
    #converte lista crua com bits dos binários em lista com vetores protótipos/acidentes
        #separa listona de bits em listas com cada binário (bit/variavel)
        bin = []
        binarios = []
        for i in range(len(self.fenotipo)):
            if i % self.bitsPorVar == self.bitsPorVar-1:
                binarios.append(bin);
                bin = []
            bin.append(self.fenotipo[i])
        binsList = [[str(bit) for bit in binList] for binList in binarios]

        #converte lista de listas de bits em protótipo decimal
        decimals = self.gray(binsList)

        #divide cada fenotipo por acidente
        acidentes = []
        acidente = []
        for i in range(len(decimals)):
            acidente.append(decimals[i])
            if i % self.numVariaveis == self.numVariaveis-1:
                acidentes.append(acidente);
                acidente = []
        return acidentes

    def mutacionar(self,portao,delta,epsilon,maiorScore,melhor):
        for qubit in self.dna:
            qubit.atualizar(self.fitness, portao, delta, epsilon, maiorScore, melhor)
        self.fenotipo = self.decodificar()
        self.prototipos = self.transcrever()
        self.fitness = 0
        return

    def props(self):
        resposta = {
            'id': self.id,
            'dna': [qubit.props() for qubit in self.dna],
            'fenotipo': self.fenotipo,
            'prototipos': self.prototipos,
            'fitness': self.fitness,
        }
        return resposta

In [ ]:
v=np.array([0,2,3,1,0,-1,2,4,5])
print(v.argmin())

5


In [24]:
portoes = ['Rx','Ry','Rz','PauliX','PauliY','PauliZ','H','S','T']
condicoes = ['blackout','loca','sgtr']
variaveis = ['vazDoNucleo','tempPQuente','tempPFria','vazNoNucleo','GVLarga','GVEstreita','pressaoGV','vazAguaAlimentacao','vazVapor','vazRuptura','vazPrimario','tempo','pressaoPrimario','potTermica','potNuclear','subresfriamento','nivelPressurizador','tempPrimario']

class Populacao():
    def __init__(self,tamanho=100):
        self.geracao = 0
        self.cromos = [Cromossomo(i) for i in range(tamanho)]
        self.adaptados = []
        self.duracao = 61 #segundos por assinatura de estado
        self.assinaturas = self.configurar_assinaturas()
        self.maiorAcerto = 0
        print('População pronta')

    def __str__(self):
        return str([str(adaptado) for adaptado in self.adaptados])

    def props(self):
        response = {
            'geracao': self.geracao,
            'cromos': [cromo.props() for cromo in self.cromos],
            'adaptados': self.adaptados,
            'duracao': self.duracao,
            'assinaturas': self.assinaturas,
        }
        return response

    def distancia(self,v1,v2):
        a = np.array(v1)
        b = np.array(v2)
        return np.sqrt(np.sum(((a - b) ** 2)))

    def get_assinaturas(self):
        res = requests.get('https://drive.google.com/uc?id=1KooJzeuq0yRSdcT1OWwbKGR3VgOsvsDZ')
        return list(res.json().values())

    def normalizar(self,inputs):
        #min-max equalizado
        mx = max(inputs)
        mn = min(inputs)
        if mx == mn:
            return [0 for input in inputs]
        return [(x-mn)/(mx-mn) for x in inputs]


    def testar(self,cromo):
        blackout,loca,sgtr = self.assinaturas#[[estado[variavel for variavel in variaveis] for estado in acidente] for acidente in self.assinaturas]
        # blackout_prot,loca_prot,sgtr_prot = cromo.prototipos
        estado = lambda assinatura, instante: [variavel[instante] for variavel in assinatura]
        for acidente in range(len(cromo.prototipos)):
            erro = lambda possibilidade: self.distancia(cromo.prototipos[acidente],possibilidade)
            for seg in range(self.duracao):
                distancias = [erro(estado(blackout,seg)),erro(estado(loca,seg)),erro(estado(sgtr,seg))]
                if np.array(distancias).argmin() == acidente:
                    cromo.fitness +=1
        return

    def configurar_assinaturas(self):
        assinaturas = self.get_assinaturas()
        print(assinaturas)
        #temporal_normalizada = [[self.normalizar([float(estado[variavel]) for variavel in variaveis]) for estado in acidente] for acidente in assinaturas]#[float(estado[variavel]) for variavel in estado]) for estado in acidente] for acidente in assinaturas]
        temporal_normalizada = [[self.normalizar([float(estado[var]) for estado in acidente]) for var in variaveis] for acidente in assinaturas]#[[self.normalizar([float(estado[variavel]) for variavel in variaveis]) for estado in acidente] for acidente in assinaturas]#[float(estado[variavel]) for variavel in estado]) for estado in acidente] for acidente in assinaturas]
        print(len([[variavel[1] for variavel in acidente] for acidente in temporal_normalizada]),[[variavel[1] for variavel in acidente] for acidente in temporal_normalizada])
        return temporal_normalizada

    def avaliar(self):
        for cromo in self.cromos:
            self.testar(cromo)

            if cromo.fitness >= self.maiorAcerto:
                self.maiorAcerto = cromo.fitness
                self.adaptados.append({"id": cromo.id, "acertos": cromo.fitness, "fenotipos": cromo.fenotipo})
        return

    def procriar(self,portao,delta,epsilon):
        for cromo in self.cromos:
            cromo.mutacionar(portao, delta, epsilon, self.maiorAcerto, self.adaptados[-1])
        return


In [25]:
def qea(genMax,delta,portao,epsilon):
    populacao = Populacao()
    print(populacao.cromos)
    print(populacao.props)

    path = './drive/MyDrive/TCC/{portao}'.format(portao=portao)
    #os.mkdir(path)
    while populacao.geracao < genMax:
        populacao.avaliar()
        with io.open('./drive/MyDrive/TCC/{portao}/geracao-{geracao}.json'.format(portao=portao,geracao=populacao.geracao),'w+') as persistor:
            json.dump(populacao.props(),persistor)
        populacao.geracao +=1
        populacao.procriar(portao,delta,epsilon)
        populacao.adaptados = []
    #     print('Adaptados:',populacao.adaptados.length)
    #     # iDB.persist(populacao,"qubits")
    #     # iDB.persist({melhores: populacao.adaptados,geracao:populacao.geracao},"adaptados")
    #     postMessage({eventype:"nextGen"})
    #     populacao.cromos.forEach(cromo => cromo.learn(portao, delta,epsilon,populacao.maiorScore,populacao.adaptados[-1]))
    #     print('learnt!')
    #     print(populacao)


qea(10,0.05,'Ry',0.01)

[[{'vazDoNucleo': '+.105223E+03', 'tempPQuente': '+.324465E+03', 'tempPFria': '+.291456E+03', 'vazNoNucleo': '+.504184E+05', 'GVLarga': '+.500000E+02', 'GVEstreita': '+.625153E+02', 'pressaoGV': '+.689480E+01', 'vazAguaAlimentacao': '+.527767E+03', 'vazVapor': '+.527767E+03', 'vazRuptura': '+.000000E+00', 'vazPrimario': '+.105223E+03', 'tempo': '+.000000E+00', 'pressaoPrimario': '+.158000E+02', 'potTermica': '+.100000E+03', 'potNuclear': '+.100000E+03', 'subresfriamento': '+.218620E+02', 'nivelPressurizador': '+.591491E+02', 'tempPrimario': '+.307959E+03'}, {'vazDoNucleo': '+.105390E+03', 'tempPQuente': '+.324463E+03', 'tempPFria': '+.291457E+03', 'vazNoNucleo': '+.504183E+05', 'GVLarga': '+.500078E+02', 'GVEstreita': '+.625169E+02', 'pressaoGV': '+.689413E+01', 'vazAguaAlimentacao': '+.527890E+03', 'vazVapor': '+.528396E+03', 'vazRuptura': '+.000000E+00', 'vazPrimario': '+.105390E+03', 'tempo': '+.100000E+01', 'pressaoPrimario': '+.157999E+02', 'potTermica': '+.100005E+03', 'potNuclea

In [ ]:
print([[[float(estado[variavel]) for variavel in estado] for estado in acidente]for acidente in assinaturas])

[[[105.223, 324.465, 291.456, 50418.4, 50.0, 62.5153, 6.8948, 527.767, 527.767, 0.0, 105.223, 0.0, 15.8, 100.0, 100.0, 21.862, 59.1491, 307.959], [105.39, 324.463, 291.457, 50418.3, 50.0078, 62.5169, 6.89413, 527.89, 528.396, 0.0, 105.39, 1.0, 15.7999, 100.005, 100.005, 21.8967, 59.1357, 307.942], [99.4508, 325.009, 291.5, 50044.8, 49.0423, 62.2777, 6.94732, 34.5845, 354.461, 0.0, 99.4663, 2.0, 15.4866, 11.9731, 11.9757, 30.3013, 54.6116, 303.144], [92.9815, 324.302, 291.91, 49535.0, 45.487, 61.7591, 7.18567, 34.0993, 152.994, 0.0, 93.0863, 3.0, 15.153, 10.2247, 10.2257, 38.5294, 49.9259, 298.083], [87.1439, 320.759, 292.337, 49044.1, 39.7598, 61.2484, 7.41172, 33.5098, 210.368, 0.0, 87.3292, 4.0, 14.9126, 9.05521, 9.05586, 41.9348, 46.6158, 295.654], [81.9561, 316.315, 292.692, 48593.8, 34.8187, 60.8078, 7.52939, 33.2219, 120.206, 0.0, 82.1691, 5.0, 14.7608, 8.18622, 8.18665, 43.4224, 44.5467, 294.457], [77.349, 312.096, 292.93, 48191.6, 29.3472, 60.3384, 7.6311, 32.9721, 98.3528, 0.0

In [ ]:
res = requests.get('https://drive.google.com/uc?id=1KooJzeuq0yRSdcT1OWwbKGR3VgOsvsDZ')
assinaturas = list(res.json().values())
variaveis = ['vazDoNucleo','tempPQuente','tempPFria','vazNoNucleo','GVLarga','GVEstreita','pressaoGV','vazAguaAlimentacao','vazVapor','vazRuptura','vazPrimario','tempo','pressaoPrimario','potTermica','potNuclear','subresfriamento','nivelPressurizador','tempPrimario']
# print(len([[[estado[variavel]for variavel in variaveis]for estado in acidente]for acidente in assinaturas]))
# print(list(map(lambda lists: len(lists),[[[estado[variavel]for variavel in variaveis]for estado in acidente]for acidente in assinaturas])))
# print(list(map(lambda lists: [len(lst) for lst in lists],[[[estado[variavel]for variavel in variaveis]for estado in acidente]for acidente in assinaturas])))
# print(len(list(map(lambda lists: [len(lst) for lst in lists],[[[estado[variavel]for variavel in variaveis]for estado in acidente]for acidente in assinaturas]))))
# print([[[estado[variavel]for variavel in variaveis]for estado in acidente]for acidente in assinaturas])
def normalizar(inputs):
        #min-max equalizado
        mx = max(inputs)
        mn = min(inputs)
        if mx == mn:
            return [0 for input in inputs]
        return [(x-mn)/(mx-mn) for x in inputs]

print([[normalizar([float(estado[var]) for estado in acidente]) for var in estado] for acidente in assinaturas])
#pega valores individualmente
# new = [[],[],[]]
# for acidente in assinaturas:
#     # for variavel in variaveis:
#         for estado in acidente:
#             for variavel in estado:
#                 print(variavel,len([estado[variavel] for estado in acidente]))
#                 print(variavel,[float(estado[variavel]) for estado in acidente])
#                 print(max([float(estado[variavel]) for estado in acidente]),[normalizar([float(estado[variavel]) for estado in acidente])for acidente in assinaturas])
#                 print(len([[float(estado[variavel]) for estado in acidente]for acidente in assinaturas]))
#                 print(variavel,[[float(estado[variavel]) for estado in acidente]for acidente in assinaturas])

3294
